# GodelReplay: PermutedMNIST Benchmark
### GodelPlugin (Fisher-scaled EWC-DR) + Avalanche Replay — Two-Layer Continual Learning

> **Hypothesis:** GodelReplay (Replay + GodelPlugin) achieves lower catastrophic forgetting
> than either Replay-only or EWC-only on PermutedMNIST (10 tasks).

| Strategy | Mechanism | Expected Forgetting |
|----------|-----------|---------------------|
| Naive | None (sanity baseline) | ~90% |
| Replay-only | Past-task sample buffer (mem=500) | ~8–12% |
| EWC-only (GodelPlugin) | Fisher-scaled regularization | ~31.5% (reproduces prior result) |
| **GodelReplay** | **Replay + GodelPlugin** | **< Replay-only (target)** |

**Part of the Two-Layer GodelAI Architecture:**
```
Training-time  →  GodelAI / GodelReplay  : Fisher Scaling + EWC-DR + Replay
Inference-time →  GodelAI-Lite           : MemPalace + MACP + GIFP
Together       →  Complete memory system for continual AI
```

*C-S-P: Compression → State → Propagation | YSenseAI Ecosystem 2026*

## 1. Install Dependencies

In [ ]:
import subprocess, sys

def _install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

_install('avalanche-lib', 'torch', 'torchvision')
print('avalanche-lib installed.')


## 2. Load GodelAI Repository

In [ ]:
import subprocess, os, sys

GODELAI_DIR = '/kaggle/working/godelai-repo'

if not os.path.exists(GODELAI_DIR):
    print('Cloning creator35lwb-web/godelai...')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/creator35lwb-web/godelai.git', GODELAI_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('Clone error:', result.stderr)
        raise RuntimeError('Failed to clone godelai repo. Ensure internet is enabled.')
    print('Cloned successfully.')
else:
    print('godelai-repo already present.')

if GODELAI_DIR not in sys.path:
    sys.path.insert(0, GODELAI_DIR)

# Verify core imports
from godelai.avalanche_plugin import GodelPlugin
from godelai.strategies.godel_replay import create_godel_replay_strategy
print('GodelPlugin and GodelReplay strategy imported successfully.')


## 3. Experiment Configuration

In [ ]:
import torch

CONFIG = {
    'n_experiences': 10,
    'seed': 42,
    'train_epochs': 5,
    'train_mb_size': 128,
    'eval_mb_size': 256,
    'lr': 0.001,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'ewc_lambda': 400.0,
    'fisher_scaling': 'global_max',
    'propagation_gamma': 2.0,
    't_score_window': 50,
    'mem_size': 500,
}

print('Configuration:')
for k, v in CONFIG.items():
    print(f'  {k:<22}: {v}')
print(f'\nGPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## 4. Run — 4-Strategy Comparison
> Each strategy trains on 10 PermutedMNIST tasks sequentially.
> After each task, all prior test streams are evaluated.
> Estimated runtime: 20–40 min on T4 GPU.

In [ ]:
import sys
sys.path.insert(0, GODELAI_DIR)

from experiments.permutedmnist_godelreplay import run_strategy, CONFIG

strategies = ['naive', 'replay_only', 'ewc_only', 'godel_replay']
results = []

for name in strategies:
    r = run_strategy(name, CONFIG)
    results.append(r)
    print(f'\n  -> {name}: acc={r["final_accuracy"]}, forgetting={r["avg_forgetting"]}')

print('\nAll strategies complete.')


## 5. Results

In [ ]:
print('\n' + '='*70)
print('  GODELREPLAY RESULTS — PermutedMNIST (10 tasks, seed=42)')
print('='*70)
print(f'  {"Strategy":<20} {"Final Acc":>12} {"Avg Forgetting":>16}')
print(f'  {"-"*20} {"-"*12} {"-"*16}')

for r in results:
    acc  = f'{r["final_accuracy"]:.4f}'  if r['final_accuracy']  is not None else 'N/A'
    forg = f'{r["avg_forgetting"]:.4f}'  if r['avg_forgetting']  is not None else 'N/A'
    print(f'  {r["strategy"]:<20} {acc:>12} {forg:>16}')

print('='*70)

replay_forg      = next((r['avg_forgetting'] for r in results if r['strategy'] == 'replay_only'),    None)
godelreplay_forg = next((r['avg_forgetting'] for r in results if r['strategy'] == 'godel_replay'),   None)
ewc_forg         = next((r['avg_forgetting'] for r in results if r['strategy'] == 'ewc_only'),        None)

if replay_forg and godelreplay_forg and replay_forg > 0:
    delta_pct = (replay_forg - godelreplay_forg) / replay_forg * 100
    verdict = 'HYPOTHESIS CONFIRMED' if godelreplay_forg < replay_forg else 'HYPOTHESIS REJECTED'
    print(f'\n  GodelReplay vs Replay-only : {delta_pct:+.1f}% forgetting reduction')
    print(f'  GodelReplay vs EWC-only    : {"-" if not ewc_forg else f"{(ewc_forg - godelreplay_forg)/ewc_forg*100:+.1f}%" } forgetting')
    print(f'  Verdict: {verdict}')


## 6. Ecosystem Connection

**What this experiment proves:**

GodelAI operates at two layers, both validated:

| Layer | System | Mechanism | Result |
|-------|--------|-----------|--------|
| **Training-time** | GodelReplay | Replay + Fisher-scaled EWC-DR | PermutedMNIST result above |
| **Inference-time** | GodelAI-Lite | MemPalace + MACP + GIFP | +31.2% overall, 3/3 memory retention |

**C-S-P maps to both layers identically:**

| C-S-P Stage | Training (GodelReplay) | Inference (GodelAI-Lite) |
|-------------|----------------------|------------------------|
| Compression | Fisher Information Matrix | extract_facts() |
| State | EWC-DR penalty + old params | godelai_memory.json |
| Propagation | Replay buffer samples | Portable JSON across models |

> *"Intelligence can scale through memory, not just parameters."*  
> — YSenseAI Ecosystem, GodelAI C-S-P Framework

---
**References:**
- [GodelAI Framework — Zenodo DOI 10.5281/zenodo.18048374](https://zenodo.org/records/18048374)
- [GodelAI GitHub](https://github.com/creator35lwb-web/godelai)
- [GodelAI-Lite Kaggle Notebook](https://www.kaggle.com/code/creator35lwb/godelai-lite-memory-for-gemma-4)
- Kirkpatrick et al., Overcoming catastrophic forgetting in neural networks, 2017
- Rebuffi et al., iCaRL: Incremental Classifier and Representation Learning, 2017

*Experiment by creator35lwb | FLYWHEEL TEAM | MACP v2.3.1*